# Job D: T4 frame-only substitution scores on six models, main + held-out, one A100-40GB

**Run all cells.** Section 6 runs the tokenizer checks for every model (CPU only). The queue cell (section 8)
then works through all 12 model × set items in the note's delivery order.
Each item is **resumable**: after a Colab disconnect, re-run all cells and the queue picks up where it
stopped. A finished item is skipped once its deliverable is on Drive.

| tier | due | items |
|---|---|---|
| 1 | Sun 4 Oct | Qwen3-0.6B main + held-out, Qwen3-1.7B main, Llama-3.1-8B main, Qwen3-8B main |
| 2 | Tue 6 Oct | GPT-2 main + held-out, Qwen3-1.7B held-out, OLMo-3-7B main |
| 3 | Thu 8 Oct | held-out for Qwen3-8B, Llama-3.1-8B, OLMo-3-7B |

For each item the notebook:
1. loads the model at the note's revision, resolved to the full commit sha, in the note's dtype (float32 up to 2B, bfloat16 above);
2. sizes the batches for this GPU automatically;
3. runs the **checks**, and **stops** if a blocking one fails:
   - the tokenizer checks (section 6);
   - the engine against the unmodified `ref.t4_rows`;
   - the anchor (Qwen3-0.6B, GPT-2);
   - job A's file (Qwen3-0.6B main: fields 1–7, on the first 50 sentences before the run and on every row after it);
   - the 1b cache (Qwen3-1.7B, Llama-3.1-8B);
   - causality, reported only;
4. scores every sentence with reference **v4**: 8 numbers per row, the last being `first_sub`;
5. writes `tree_t4_<model>_<set>.jsonl.gz` and `checks_<model>_<set>.json` and **copies both to Drive at once**, so each file can be sent as soon as it is done.

**Before you start:** add `HF_TOKEN` as a Colab secret with notebook access. Llama-3.1-8B is gated, so accept its licence
on the model page first. The Drive folder `m1_llms_analyzer/tree_runner_job_d/` already holds the three 1b caches the
checks need.

## 1 · Bootstrap — clone the repo

In [ ]:
#@title Clone (or update) the repository { display-mode: "form" }
import base64, json, os, subprocess, sys, textwrap, urllib.error, urllib.request
from pathlib import Path

REPO_OWNER  = "SalmonSung"
REPO_NAME   = "m1_llms_analyzer"
REPO_BRANCH = "main"   #@param {type:"string"}

CLEAN_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
NEW_PAT_URL = "https://github.com/settings/personal-access-tokens/new"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _colab_secret(name):
    """Read a Colab secret, returning None if it is absent or access is denied."""
    if not _in_colab():
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as exc:
        print(f"  (Colab secret {name!r} unavailable: {type(exc).__name__})")
        return None


def _clean(token):
    """Strip whitespace and stray quotes -- by far the most common paste error."""
    if not token:
        return None
    return token.strip().strip('"').strip("'").strip() or None


def _token_kind(token):
    """Name the token type from its prefix, without revealing the value."""
    for prefix, kind in (
        ("github_pat_", "fine-grained PAT"),
        ("ghp_", "classic PAT"),
        ("gho_", "OAuth token"),
        ("ghs_", "App installation token"),
        ("ghu_", "user-to-server token"),
    ):
        if token.startswith(prefix):
            return kind
    return "UNRECOGNISED PREFIX"


def _api(path, token):
    """GET api.github.com/<path> with the token. Raises urllib.error.HTTPError."""
    request = urllib.request.Request(
        f"https://api.github.com/{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


def _auth_config(token):
    """Auth as a per-command git config value, so it never touches .git/config or a URL."""
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraHeader=AUTHORIZATION: basic {basic}"


def _run(cmd, token=None):
    """Run git, redacting the auth header and the token from anything printed."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr or result.stdout
        if token:
            message = message.replace(token, "***")
        shown = " ".join("<auth>" if "extraHeader" in c else c for c in cmd)
        raise RuntimeError(f"git failed: {shown}\n{message}")
    return result.stdout.strip()


def _preflight(token):
    """Verify the token before git runs, so failures name their actual cause.

    A bare `git clone` failure says only 'Invalid username or token', which covers an
    expired token, a typo, a missing repo grant, and un-authorised SSO alike. These two
    API calls tell those apart.
    """
    kind = _token_kind(token)
    print(f"GITHUB_TOKEN: {len(token)} chars, looks like a {kind}.")
    if kind == "UNRECOGNISED PREFIX":
        print("  Warning: GitHub tokens start with github_pat_, ghp_, gho_, ghs_ or ghu_.")
        print("  If you pasted an account password or an SSH key, that will not work here.")

    try:
        me = _api("user", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise SystemExit(textwrap.dedent(f"""
                GitHub rejected this token (401 Unauthorized). The token itself is bad --
                this is not a permissions problem. Most likely one of:

                  * it has expired (fine-grained PATs expire, 30 days by default);
                  * it was revoked or regenerated;
                  * the secret holds something that is not a token (an account password
                    will never work -- GitHub removed password auth for git);
                  * it was truncated or mangled when pasted.

                Fix: create a new token at
                  {NEW_PAT_URL}
                  - Resource owner: {REPO_OWNER}
                  - Repository access: only select repositories -> {REPO_NAME}
                  - Permissions: Repository permissions -> Contents -> Read-only
                Then in Colab: key icon in the left sidebar -> edit GITHUB_TOKEN, paste the
                new value with no quotes and no trailing spaces, keep 'Notebook access' on,
                and re-run this cell.
            """).strip())
        if exc.code == 403:
            raise SystemExit(textwrap.dedent(f"""
                GitHub returned 403 for this token. Usually either a rate limit, or the
                token needs SAML SSO authorisation for the '{REPO_OWNER}' organisation.
                If {REPO_OWNER} is an org with SSO, open your token's settings page and
                click 'Configure SSO' -> Authorize.

                Original error: {exc}
            """).strip())
        raise

    print(f"  Authenticates as: {me.get('login')}")

    try:
        repo = _api(f"repos/{REPO_OWNER}/{REPO_NAME}", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            login = me.get("login")
            not_owner = (
                f"\n                  * you are {login}, but the repo belongs to "
                f"{REPO_OWNER} and you are not a collaborator on it;"
                if login and login.lower() != REPO_OWNER.lower()
                else ""
            )
            raise SystemExit(textwrap.dedent(f"""
                The token is valid (you are {login}), but it cannot see
                {REPO_OWNER}/{REPO_NAME}. GitHub returns 404 rather than 403 for a private
                repo a token has no grant on, so this means one of:

                  * the token's 'Repository access' does not include {REPO_NAME};
                  * it lacks the 'Contents: Read-only' repository permission;{not_owner}
                  * the owner or name is misspelled (both are case-sensitive).

                Fix: open {NEW_PAT_URL} (or edit the existing token), grant this
                repository and Contents: Read-only, then re-run this cell.
            """).strip())
        raise

    print(f"  Repo access:      OK ({'private' if repo.get('private') else 'public'})")
    return True


_TOKEN = _clean(_colab_secret("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN"))

if not _in_colab() and Path("pyproject.toml").exists():
    # Running from a local checkout -- nothing to clone.
    REPO_DIR = Path.cwd()
    print(f"Local checkout detected: {REPO_DIR}")
else:
    REPO_DIR = Path("/content") / REPO_NAME if _in_colab() else Path.cwd() / REPO_NAME
    if not _TOKEN:
        raise SystemExit(textwrap.dedent(f"""
            GITHUB_TOKEN is not set, and {REPO_OWNER}/{REPO_NAME} is private.
              1. Create a fine-grained PAT at {NEW_PAT_URL}
                 - Resource owner: {REPO_OWNER}
                 - Repository access: only select repositories -> {REPO_NAME}
                 - Permissions: Contents -> Read-only
              2. Colab left sidebar -> key icon -> add a secret named GITHUB_TOKEN
              3. Turn on 'Notebook access' for it, then re-run this cell.
        """).strip())

    _preflight(_TOKEN)

    # Auth travels as a per-command header, never in the URL. Nothing is written to
    # .git/config, so there is no token left on disk to scrub afterwards.
    _AUTH = _auth_config(_TOKEN)
    try:
        if REPO_DIR.exists():
            print(f"\nRepo already present at {REPO_DIR}; updating...")
            _run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", CLEAN_URL], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "-c", _AUTH, "fetch", "origin", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], _TOKEN)
        else:
            print(f"\nCloning into {REPO_DIR} ...")
            _run(["git", "-c", _AUTH, "clone", "--branch", REPO_BRANCH, "--depth", "1",
                  CLEAN_URL, str(REPO_DIR)], _TOKEN)
    except RuntimeError as exc:
        # The API accepted the token but git did not -- rare, and worth naming, because
        # the obvious readings (bad token, missing grant) were just ruled out above.
        raise SystemExit(textwrap.dedent(f"""
            {exc}

            The token passed the API preflight above, so it is valid and can see this
            repo -- the failure is in the git transport itself. Things to check:

              * branch '{REPO_BRANCH}' exists on the remote (a typo in REPO_BRANCH gives
                'Remote branch not found');
              * a stale {REPO_DIR} from an earlier run: delete it and re-run this cell;
              * a corporate proxy or VPN intercepting HTTPS to github.com.
        """).strip())
    print("Done. Remote is", CLEAN_URL, "(no credentials stored on disk).")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Commit:", _run(["git", "rev-parse", "--short", "HEAD"]))
del _TOKEN  # do not leave the token bound in the notebook namespace

## 2 · Dependencies

Installed with `--upgrade-strategy only-if-needed` so Colab's preinstalled,
CUDA-matched `torch` is **kept** rather than reinstalled (a torch swap costs several
minutes and can break GPU support).

In [ ]:
#@title Install dependencies
import subprocess, sys

print("Installing (quiet; ~30s on a cold runtime)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", "-e", "."],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] or "(no output)")
if result.returncode != 0:
    print(result.stderr[-3000:], file=sys.stderr)
    raise SystemExit("Dependency installation failed -- see the error above.")

# Make the freshly installed package importable in this already-running kernel.
import importlib, site
importlib.reload(site)
for module in [m for m in list(sys.modules) if m.startswith("m1_analyzer")]:
    del sys.modules[module]

import m1_analyzer
print("m1_analyzer", m1_analyzer.__version__, "ready")

## 3 · Environment report

In [ ]:
#@title What am I running on?
import torch, transformers, numpy, platform
from m1_analyzer import in_colab, resolve_hf_token

print(f"python        : {platform.python_version()}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"numpy         : {numpy.__version__}")
print(f"in Colab      : {in_colab()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print(f"CUDA          : {torch.version.cuda}")
else:
    print("GPU           : none -- running on CPU.")
    print("                Runtime -> Change runtime type -> T4 GPU for anything above ~1B params.")

# Only reports presence. The token value is never printed.
print(f"HF_TOKEN      : {'found' if resolve_hf_token() else 'not set (fine for ungated models)'}")

# The dtype the scorer will run in. A T4 gets float16 (its bfloat16 is emulated and slow);
# Ampere and newer get bfloat16; CPU gets float32.
from m1_analyzer.utils.device import resolve_device, resolve_dtype
print(f"scoring dtype : {str(resolve_dtype(resolve_device('auto'), 'auto')).replace('torch.', '')}")

## 4 · Configuration

`ITEMS` is `all` (the queue in delivery order), or a comma list such as `qwen3-8b:main,gpt2:heldout`. `LIMIT` > 0
scores only that many sentences per item as a dry run, and writes no deliverable. Batch sizes are found
automatically for each model (section 7). `GROUP_SEQS` is how many substituted sentences share one packing pool.
Larger pools pack better, but a disconnect then loses more work.

In [ ]:
#@title Run configuration { display-mode: "form" }
ITEMS          = "all"    #@param {type:"string"}
LIMIT          = 0        #@param {type:"integer"}
GROUP_SEQS     = 20000    #@param {type:"integer"}
MIRROR_MINUTES = 15       #@param {type:"number"}
STOP_ON_FAIL   = True     #@param {type:"boolean"}
DRIVE_DIR      = "/content/drive/MyDrive/m1_llms_analyzer/tree_runner_job_d"  #@param {type:"string"}
OUTPUT_DIR     = "outputs/job_d"                                               #@param {type:"string"}
HF_CACHE_DIR   = ""       #@param {type:"string"}
TOKENIZER_WORKERS = 0     #@param {type:"integer"}

import json
import shutil
from pathlib import Path

from m1_analyzer import in_colab
from m1_analyzer.experiments import job_d, tree_runner
from m1_analyzer.experiments import tree_runner_ref_v4 as ref

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)

if ITEMS.strip() == "all":
    QUEUE = list(job_d.QUEUE)
else:
    tiers = {(m, s): t for m, s, t in job_d.QUEUE}
    QUEUE = []
    for item in ITEMS.split(","):
        slug, set_name = item.strip().split(":")
        if (slug, set_name) not in tiers:
            raise SystemExit(f"{item!r} is not a job D item; choose from {sorted(tiers)}")
        QUEUE.append((slug, set_name, tiers[(slug, set_name)]))


def drive_dir():
    """The Drive folder, mounted on first use; None outside Colab."""
    if not in_colab():
        return None
    target = Path(DRIVE_DIR)
    if not Path("/content/drive").exists():
        from google.colab import drive  # noqa: F401 -- Colab only
        drive.mount("/content/drive")
    target.mkdir(parents=True, exist_ok=True)
    return target


def restore(path: Path) -> bool:
    """Copy `path` back from Drive when it is missing locally. True if it is now present."""
    if path.exists():
        return True
    drive = drive_dir()
    if drive is not None and (drive / path.name).exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(drive / path.name, path)
        print(f"restored {path.name} from Drive")
    return path.exists()


def mirror_path(path: Path):
    drive = drive_dir()
    return None if drive is None else drive / path.name


def push(path: Path) -> None:
    """Copy a finished file to Drive now."""
    if mirror_path(path):
        shutil.copy(path, mirror_path(path))
        print(f"  -> Drive: {mirror_path(path)}")


def save_json(path: Path, payload) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(payload, indent=1), encoding="utf-8")
    tmp.replace(path)
    push(path)


print(f"reference v4 : sha256 {job_d.reference_sha256()[:12]}…  unmodified={job_d.reference_unmodified()}")
if not job_d.reference_unmodified():
    raise SystemExit("tree_runner_ref_v4.py differs from the package's; restore it before running.")
SENTENCES = {s: job_d.load_sentences(set_name=s) for s in job_d.SETS}   # sha256 checked against the note
ALL_SENTENCES = SENTENCES["main"] + SENTENCES["heldout"]
ANCHOR = job_d.load_anchor()
print(f"sentences    : main {len(SENTENCES['main'])}, heldout {len(SENTENCES['heldout'])}  (sha256 {job_d.SENTENCES_SHA256[:12]}…)")
print(f"queue        : {[f'{m}:{s}' for m, s, _ in QUEUE]}")
print(f"drive        : {DRIVE_DIR if in_colab() else '(not in Colab)'}")

## 5 · Smoke test (no download, under a minute)

The engine, the resumable loop, the checks and the deliverable, run on a random 4-layer GPT-2 with the first
sentences of the file. It checks that the engine gives `ref.t4_rows`'s values and that a resumed run adds nothing.
The numbers themselves mean nothing.

In [ ]:
#@title Smoke test on a tiny model
import tempfile, time
from m1_analyzer.testing import build_tiny_local_model

started = time.perf_counter()
sents = SENTENCES["main"][:3]
vocab = sorted({w for s in sents for w in s["words"]} | {w for f in ref.FILLERS if f != "<del>" for w in f.split()})
extra = vocab + [w[:1].upper() + w[1:] for w in vocab] + [".", "?", "!"]
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    path = build_tiny_local_model(tmp / "tiny", extra_vocab=extra, max_positions=128)
    stok, smdl, sbos, _, smoke = tree_runner.load_model_f32(path, revision=None, device="cpu", bos_token=None)
    budget = job_d.TokenBudget(2000)
    meta = {"model_id": "tiny", "revision": None, "dtype": "float32", "bos_id": sbos, "bos_token": "[EOS]"}
    kw = dict(set_name="main", model_meta=meta, device="cpu", budget=budget, cache_path=tmp / "w.jsonl",
              show_progress=False, group_seqs=500)
    rows = job_d.run_item(stok, smdl, sbos, sents, **kw)
    assert job_d.run_item(stok, smdl, sbos, sents, **kw) == rows
    refr = [dict(id=s["id"], end=s["end"], **dict(zip(("orig", "rows"), ref.t4_rows(stok, smdl, sbos, s["words"], s["keys"], s["end"]))))
            for s in sents]
    eq = job_d.check_equivalence(rows, refr, dtype="float32")
    assert eq["pass"], eq
    checks = {"engine_equivalence": eq, "complete": job_d.check_complete(rows, sents)}
    m = job_d.item_meta(job_d.MODELS_D["gpt2"], "main", meta, n_sentences=3, n_spans=0, wall_seconds=1.0,
                        checks=checks, budget=budget)
    job_d.finalize_item(rows, sents, tmp / "t.jsonl.gz", m)
    smoke.unload()
print(f"engine vs ref: max |diff| total {eq['abs_diff']['total']['max']:.1e} nats on {eq['n_rows']} rows; "
      f"{time.perf_counter() - started:.1f}s")
print("Smoke test passed.")

## 6 · Tokenizer checks for every model (CPU only, no GPU needed)

The note's `tokenizer_checks(tok, rows_of_that_set)`, for every model and set in the queue. **Llama-3.1 goes
first**, because the requesters ran every other tokenizer themselves and want its counts first.
- **Gates:** `is_fast` true; `special_in_text`, `roundtrip_fail` and `offsets_bad` all 0. **If one fails, the cell
  stops.** Tell the requesters; don't work around it.
- **Reported, not gates:** `frame_pre_short`, `frame_suf_short`, `end_merge`.

Only tokenizers are loaded here. The work is split over `TOKENIZER_WORKERS` CPU processes (0 = all but one).
The counts are summed, which gives exactly the single-core result. The results are cached per tokenizer (by the
sha256 of its JSON) and per set, in `tokenizer_<sha12>_<set>.json` on Drive. The three Qwen3 models share one
tokenizer, so each set is computed only once for them. The queue reuses these results.

In [ ]:
#@title Run the tokenizer checks (Llama-3.1 first)
import time

TOK_CHECKS = {}


def tokenizer_block(spec, set_name, tok=None):
    """The gated tokenizer_checks block for one model and set, cached per tokenizer and set."""
    if (spec.slug, set_name) in TOK_CHECKS:
        return TOK_CHECKS[(spec.slug, set_name)]
    if tok is None:
        tok, _ = job_d.load_tokenizer_d(spec, cache_dir=HF_CACHE_DIR or None)
    sha = job_d.tokenizer_sha(tok)
    path = OUT / job_d.tokenizer_check_name(sha, set_name)
    if restore(path):
        counts = json.loads(path.read_text())["counts"]
        source = "cached"
    else:
        started = time.perf_counter()
        counts = job_d.tokenizer_checks_parallel(tok, SENTENCES[set_name], workers=TOKENIZER_WORKERS or None)
        source = f"computed in {time.perf_counter() - started:.0f}s"
        save_json(path, {"tokenizer_sha256": sha, "set": set_name, "first_computed_for": spec.hf_id, "counts": counts})
    block = {**job_d.tokenizer_gate(counts, tok_sha=sha, model_id=spec.hf_id), "set": set_name, "file": path.name}
    TOK_CHECKS[(spec.slug, set_name)] = block
    print(f"  {spec.slug:<12} {set_name:<8} {'PASS' if block['pass'] else 'FAIL ' + str(block['failed'])}  ({source})  "
          + ", ".join(f"{k}={counts.get(k)}" for k in ("is_fast", "n_texts", "special_in_text", "roundtrip_fail",
                                                      "offsets_bad", "frame_pre_short", "frame_suf_short", "end_merge")))
    return block


order = sorted({m for m, _, _ in QUEUE}, key=lambda m: (m != "llama3.1-8b", [q[0] for q in job_d.QUEUE].index(m)))
failed = []
for slug in order:
    spec = job_d.MODELS_D[slug]
    tok, rev = job_d.load_tokenizer_d(spec, cache_dir=HF_CACHE_DIR or None)
    print(f"{spec.hf_id} @ {rev[:12]}")
    for set_name in [s for m, s, _ in QUEUE if m == slug]:
        if not tokenizer_block(spec, set_name, tok)["pass"]:
            failed.append(f"{slug}:{set_name}")
if failed:
    raise SystemExit(f"Tokenizer gate FAILED for {failed}. Stop and send the tokenizer_*.json files to the requesters.")
print("\nAll tokenizer gates pass.")

## 7 · Status

Shows which deliverables are already on Drive, or local when running outside Colab. Re-run it at any time.

In [ ]:
#@title What is done?
def status():
    print(f"{'item':<24}{'tier':>5}  {'due':<11}{'deliverable':<14}{'checks':<10}work cache")
    for slug, set_name, tier in job_d.QUEUE:
        out, chk, work = (OUT / job_d.output_name(slug, set_name), OUT / job_d.checks_name(slug, set_name),
                          OUT / job_d.work_name(slug, set_name))
        drive = drive_dir()
        have = lambda p: p.exists() or (drive is not None and (drive / p.name).exists())
        c = "-"
        if have(chk) and restore(chk):
            c = "PASS" if json.loads(chk.read_text()).get("pass") else "FAIL"
        print(f"{slug + ':' + set_name:<24}{tier:>5}  {job_d.TIER_DUE[tier]:<11}{'DONE' if have(out) else '-':<14}{c:<10}"
              f"{'yes' if have(work) else '-'}")

status()

## 8 · Run the queue (GPU, resumable)

For every item in `QUEUE` that has no deliverable yet:
- **Model**: loaded once, then kept for the next item if it is the same model. The full revision sha, dtype and BOS are
  checked against the note and the anchor, and TF32 is off, so the float32 models compute as job A did.
- **Auto-sizing**: `calibrate_budget` measures the free memory after the weights, probes the longest sequence
  (halving on out-of-memory), times a few budgets and keeps the fastest. If the full run still runs out of memory,
  it halves that batch and shrinks the budget.
- **Checks** (stop on a blocking failure):
  - `tokenizer`: section 6's block for this model and set (computed now if section 6 was skipped).
  - `engine_equivalence`: 3 sentences (the shortest, a middle one and the longest of the first 50), scored by the unmodified `ref.t4_rows` and by the engine.
  - `anchor`: the anchor file's sentences, for Qwen3-0.6B and GPT-2.
  - `job_a_first50` (Qwen3-0.6B main): the first 50 sentences, fields 1–7 against job A's file (`tree_t4_qwen3_0.6b.jsonl.gz`, on Drive).
  - `cache`: the first 50 main sentences against the 1b cache, for Qwen3-1.7B and Llama-3.1-8B.
  - `extra_cache`: the same against Qwen3-8B's 1b cache. Reported only.
  - `causality`: reported only.
- **Full run**: the progress bar shows the budget and a live ETA in hours.
- **Stale caches**: a working cache written under an older reference (e.g. v2, without `first_sub`) is renamed
  `*.stale-<sha8>`, locally and on Drive, and the item starts again.
- **Deliverable**: final checks (`complete`, `causality_full`, and `job_a` on every row for Qwen3-0.6B main), then `tree_t4_<model>_<set>.jsonl.gz` and
  `checks_<model>_<set>.json`, both pushed to Drive immediately.

In [ ]:
#@title Run every pending item
import gc, time
from datetime import datetime, timedelta, timezone

import transformers

LOADED = {}


def ensure_model(spec):
    if LOADED.get("slug") == spec.slug:
        return LOADED
    if LOADED.get("analyzer") is not None:
        LOADED["analyzer"].unload()
        LOADED.clear()
        gc.collect()
    if spec.slug.startswith("olmo3") and not hasattr(transformers.models, "olmo3"):
        raise SystemExit(f"transformers {transformers.__version__} has no OLMo-3. Run: pip install -U transformers, "
                         "then restart the runtime and re-run all cells.")
    started = time.perf_counter()
    tok, mdl, bos, device, analyzer, meta = job_d.load_model_d(spec, anchor=ANCHOR, cache_dir=HF_CACHE_DIR or None)
    print(f"\nloaded {spec.hf_id} @ {meta['revision']}  {meta['dtype']}  BOS {meta['bos_token']!r} ({bos})  "
          f"on {device} in {time.perf_counter() - started:.0f}s")
    budget, calib = job_d.calibrate_budget(mdl, bos, device, max_len=job_d.max_seq_len(tok, ALL_SENTENCES))
    print(f"auto-sizing: {calib}")
    LOADED.update(slug=spec.slug, tok=tok, mdl=mdl, bos=bos, device=device, analyzer=analyzer, meta=meta,
                  budget=budget, calib=calib)
    return LOADED


def show(checks):
    for name, block in checks.items():
        if not isinstance(block, dict):
            continue
        state = "PASS" if block.get("pass") else ("FAIL" if "pass" in block else "reported")
        brief = {k: v for k, v in block.items() if k not in ("pass",) and not (isinstance(v, list) and not v)}
        print(f"  {name:<20} {state:<9} {json.dumps(brief)[:400]}")


def run_checks(spec, set_name, L, work):
    tok, mdl, bos, device, budget = L["tok"], L["mdl"], L["bos"], L["device"], L["budget"]
    sents = SENTENCES[set_name]
    first = sents[:job_d.CACHE_CHECK_FIRST]
    by_len = sorted(first, key=lambda s: len(s["words"]))
    eq_sents = [by_len[0], by_len[len(by_len) // 2], by_len[-1]]
    checks = {"tokenizer": tokenizer_block(spec, set_name, tok)}
    if not checks["tokenizer"]["pass"]:
        return {**checks, "pass": False}
    t = time.perf_counter()
    ref_rows = []
    for s in eq_sents:
        orig, rows = ref.t4_rows(tok, mdl, bos, s["words"], s["keys"], s["end"], batch=64, device=device)
        ref_rows.append({"id": s["id"], "end": s["end"], "orig": orig, "rows": rows})
    t_ref = time.perf_counter() - t
    t = time.perf_counter()
    fast = job_d.score_sentences(tok, mdl, bos, eq_sents, device=device, budget=budget)
    t_fast = time.perf_counter() - t
    checks["engine_equivalence"] = {**job_d.check_equivalence(fast, ref_rows, dtype=spec.dtype),
                                    "seconds_ref": round(t_ref, 1), "seconds_engine": round(t_fast, 1)}
    check_rows = list(fast)
    if spec.anchor:
        ids = {a["id"] for a in ANCHOR["models"][spec.hf_id]["sentences"]}
        arows = job_d.score_sentences(tok, mdl, bos, [s for s in ALL_SENTENCES if s["id"] in ids],
                                      device=device, budget=budget)
        checks["anchor"] = job_d.check_anchor(arows, ANCHOR, spec.hf_id)
        check_rows += arows
    eta = None
    if set_name == "main" and (spec.cache_file or spec.extra_cache_file or spec.job_a_file):
        t = time.perf_counter()
        rows50 = job_d.run_item(tok, mdl, bos, sents, set_name=set_name, model_meta=L["meta"], device=device,
                                budget=budget, cache_path=work, ids=[s["id"] for s in first], group_seqs=GROUP_SEQS,
                                mirror_path=mirror_path(work), mirror_minutes=MIRROR_MINUTES, show_progress=True)
        secs = tree_runner.wall_seconds(rows50) or (time.perf_counter() - t)
        eta = secs / job_d.cost_units(first) * job_d.cost_units(sents) / 3600
        if spec.job_a_file:
            if not restore(OUT / spec.job_a_file):
                raise SystemExit(f"{spec.job_a_file} is not in {OUT} nor on Drive ({DRIVE_DIR}).")
            checks["job_a_first50"] = job_d.check_job_a(rows50, OUT / spec.job_a_file)
        for fname, name in ((spec.cache_file, "cache"), (spec.extra_cache_file, "extra_cache")):
            if not fname:
                continue
            if not restore(OUT / fname):
                raise SystemExit(f"{fname} is not in {OUT} nor on Drive ({DRIVE_DIR}).")
            header, cache = job_d.load_span_totals(OUT / fname)
            block = job_d.check_cache(rows50, cache, first)
            block.update(cache_file=fname, cache_model=header.get("model_id"), cache_revision=header.get("revision"),
                         cache_dtype=header.get("dtype"))
            checks[name] = block
        check_rows += rows50
    else:
        eta = t_fast / job_d.cost_units(eq_sents) * job_d.cost_units(sents) / 3600
    checks["causality"] = job_d.check_causality(check_rows, spec.dtype)
    checks["pass"] = job_d.blocking_pass(checks)
    checks["token_budget"] = budget.as_meta()
    checks["calibration"] = L["calib"]
    checks["eta_hours_estimate"] = round(eta, 2)
    checks["reference_code_sha"] = job_d.REFERENCE_SHA256
    checks["engine_version"] = job_d.ENGINE_VERSION
    return checks


def retire_stale(work: Path) -> None:
    """Rename a working cache written under another reference (local and Drive); it cannot be resumed."""
    if not work.exists():
        return
    ok, written = job_d.cache_is_current(work)
    if ok:
        return
    tag = f".stale-{written}"
    work.rename(work.with_name(work.name + tag))
    if mirror_path(work) and mirror_path(work).exists():
        mirror_path(work).rename(mirror_path(work).with_name(work.name + tag))
    print(f"  {work.name} was written under reference/engine {written}; renamed to {work.name + tag} and starting again.")


def run_one(slug, set_name, tier):
    spec = job_d.MODELS_D[slug]
    sents = SENTENCES[set_name]
    out, chk, work = (OUT / job_d.output_name(slug, set_name), OUT / job_d.checks_name(slug, set_name),
                      OUT / job_d.work_name(slug, set_name))
    if restore(out):
        print(f"{slug}:{set_name} is done ({out.name}); skipping.")
        return
    print(f"\n=== {slug}:{set_name}  (tier {tier}, due {job_d.TIER_DUE[tier]}) ===")
    L = ensure_model(spec)
    restore(work)
    retire_stale(work)
    checks = json.loads(chk.read_text()) if restore(chk) else None
    if (not checks or not checks.get("pass") or checks.get("revision") != L["meta"]["revision"]
            or checks.get("reference_code_sha") != job_d.REFERENCE_SHA256
            or checks.get("engine_version") != job_d.ENGINE_VERSION):
        checks = run_checks(spec, set_name, L, work)
        checks["revision"] = L["meta"]["revision"]
        save_json(chk, checks)
    show(checks)
    if not checks["pass"]:
        msg = f"{slug}:{set_name}: a blocking check FAILED. Send {chk.name} back; do not use this model's numbers."
        if STOP_ON_FAIL:
            raise SystemExit(msg)
        print(msg)
        return
    eta = checks.get("eta_hours_estimate")
    if eta is not None:
        finish = datetime.now(timezone.utc) + timedelta(hours=eta)
        late = finish.date().isoformat() > job_d.TIER_DUE[tier]
        print(f"  rough ETA {eta:.1f} h -> about {finish:%a %d %b %H:%M} UTC"
              + ("   !! later than this tier's due date -- tell the requesters now" if late else ""))
    started = time.perf_counter()
    rows = job_d.run_item(L["tok"], L["mdl"], L["bos"], sents, set_name=set_name, model_meta=L["meta"],
                          device=L["device"], budget=L["budget"], cache_path=work, limit=LIMIT or None,
                          group_seqs=GROUP_SEQS, mirror_path=mirror_path(work), mirror_minutes=MIRROR_MINUTES)
    print(f"  {len(rows)} / {len(sents)} sentences ({(time.perf_counter() - started) / 3600:.2f} h this session, "
          f"{tree_runner.wall_seconds(rows) / 3600:.2f} h of compute in all)")
    if LIMIT:
        print("  LIMIT > 0: dry run, no deliverable.")
        return
    final = dict(checks)
    final["causality_full"] = job_d.check_causality(rows, spec.dtype)
    final["complete"] = job_d.check_complete(rows, sents)
    if not final["complete"]["pass"]:
        save_json(chk, final)
        raise SystemExit(f"{slug}:{set_name}: incomplete or non-finite rows: {final['complete']}")
    if spec.job_a_file and set_name == "main":
        restore(OUT / spec.job_a_file)
        final["job_a"] = job_d.check_job_a(rows, OUT / spec.job_a_file)
        print(f"  job A, every row: {'PASS' if final['job_a']['pass'] else 'FAIL'} {json.dumps(final['job_a'])[:300]}")
        if not final["job_a"]["pass"]:
            final["pass"] = False
            save_json(chk, final)
            raise SystemExit(f"{slug}:{set_name}: fields 1-7 do not match job A. No deliverable written; send {chk.name}.")
    meta = job_d.item_meta(spec, set_name, L["meta"], n_sentences=len(sents), n_spans=sum(len(s["keys"]) for s in sents),
                           wall_seconds=tree_runner.wall_seconds(rows), checks=final, budget=L["budget"],
                           extra={"calibration": L["calib"]})
    path = job_d.finalize_item(rows, sents, out, meta)
    save_json(chk, final)
    push(path)
    print(f"  wrote {path.name} ({path.stat().st_size / 2**20:.0f} MB); causality over 1e-3: "
          f"{final['causality_full']['n_over']} of {final['causality_full']['n_rows']:,} rows")


for slug, set_name, tier in QUEUE:
    run_one(slug, set_name, tier)
print("\nQueue finished.")
status()

## 9 · Tests (optional)

In [ ]:
#@title Run job D's tests inside Colab
import subprocess, sys

result_proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_job_d.py"], capture_output=True, text=True)
print(result_proc.stdout[-4000:])
print(result_proc.stderr[-2000:], file=sys.stderr)

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `tree_runner_ref_v4.py differs from the package's` | `git checkout src/m1_analyzer/experiments/tree_runner_ref_v4.py`. The run must use the reference unmodified. |
| `sha256 is not the package's` | `data/job_d/sentences_3161.jsonl` was changed. Restore it from the repository. |
| `revision … matches 0 commits` / `Access … denied` | Llama-3.1-8B is gated: accept the licence and set the `HF_TOKEN` secret. |
| `transformers … has no OLMo-3` | `pip install -U transformers`, restart the runtime, then run all cells again. |
| `engine_equivalence` FAIL | Stop. Send `checks_<model>_<set>.json` back. |
| `anchor` / `cache` FAIL | Stop. Send `checks_<model>_<set>.json` back, as the note asks. |
| Out of memory | This is handled automatically: the batch is halved and the budget shrinks (`oom_backoffs` in the meta). |
| `tokenizer` gate FAIL | Stop and send the `tokenizer_*.json` files. The note says not to work around it. |
| `fields 1-7 do not match job A` | Stop. Send `checks_qwen3-0.6b_main.json`; no deliverable is written. |
| `… was written under reference/engine …; renamed` | Expected once after an update: a v2 cache lacks `first_sub`, and a `d1` cache scored the originals inside batches. Neither can be resumed. |
| `was written for …=…` on resume | The working cache belongs to another model, revision or dtype. Delete `work_t4_<model>_<set>.jsonl` (locally and on Drive). |
| Colab disconnected | Re-run all cells. Finished items are skipped and the current one resumes. |